## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 165 samples and 12676 columns.
First few rows of the data:


,month,cancer_death,sample_title,subtype,tnm,Stage,grading,cohort,serpinb3 expression (high_low),SAMD11,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,17.47,1,100_67161,classical,T3N0MX,IIA,G1,UMD-3,L,1.401006,...,6.185995,3.425500,5.478883,7.337665,4.451029,4.213376,5.934091,5.553229,3.183231,5.726467
1,25.33,1,106_62660,classical,T3N0MX,IIA,G2,UMD-3,H,1.688630,...,7.288531,4.186766,6.135954,8.346814,5.244528,5.013787,6.677582,5.962565,3.735310,5.767141
2,37.50,1,108_56008,classical,T2N0MX,IB,G2-3,UMD-3,H,1.407038,...,5.814629,3.156867,5.208641,7.012588,4.239895,4.165831,5.887478,5.654674,3.207215,5.438136
3,1.77,1,109_57484,unclassified,T3N1MX,IIB,G3,UMD-3,L,2.244019,...,6.285254,3.350332,5.579558,7.288593,4.861689,4.647450,6.397146,5.950132,3.310646,6.209400
4,14.20,1,11_19927T,basal,T2N0M0,IB,G3,UMD-1,H,1.930785,...,6.156480,3.227813,5.322206,7.222433,5.324452,4.160351,5.937567,5.280425,3.065631,5.681370


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
